# 02 — Qwen2.5-Omni-3B: Prompt Development for Nepali ASR Benchmarking

**Model:** `Qwen/Qwen2.5-Omni-3B`  
**Architecture:** End-to-end multimodal (text + audio + image + video)  
**Class:** `Qwen2_5OmniForConditionalGeneration` + `Qwen2_5OmniProcessor`  
**Special:** Requires a specific `transformers` preview branch.


## 0. Install Dependencies

> ⚠️ Qwen2.5-Omni requires a **specific preview branch** of transformers. This cell installs it. **Restart runtime** after running.


In [ ]:
import subprocess, sys

def pip(pkg):
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "--upgrade", pkg])

# Qwen2.5-Omni requires this specific transformers preview
pip("git+https://github.com/huggingface/transformers@v4.51.3-Qwen2.5-Omni-preview")
pip("accelerate>=1.6")

pip("bitsandbytes>=0.45")
pip("sentencepiece")
pip("protobuf")

# Qwen-specific audio utilities
pip("qwen-omni-utils[decord]")
pip("soundfile>=0.12")
pip("librosa>=0.10")
pip("scipy")
pip("einops")
pip("timm")

# Metrics
pip("jiwer>=3.1")
pip("jsonlines")

# Data
pip("pandas")
pip("tqdm")

print("\n✅ All dependencies installed.")


## A. Experiment Configuration


In [ ]:
import os, json, torch, gc
from datetime import datetime

MODEL_ID   = "Qwen/Qwen2.5-Omni-3B"
MODEL_REV  = "main"
QUANT      = "auto"

experiment_config = {
    "model_id":       MODEL_ID,
    "model_revision": MODEL_REV,
    "quantization":   QUANT,
    "random_seed":    42,
    "audio_sr":       16000,
    "batch_size":     1,
    "max_new_tokens": 256,
    "temperature":    0.0,
    "do_sample":      False,
    "timestamp":      datetime.now().isoformat(),
}

AUDIO_BASE = "/kaggle/input/datasets/panditaadarsh/llm-bechmarking-audio"
RESULTS_DIR = "/kaggle/working/results/prompt_dev/qwen2_5_omni"
os.makedirs(RESULTS_DIR, exist_ok=True)

with open(f"{RESULTS_DIR}/run_config.json", "w") as f:
    json.dump(experiment_config, f, indent=2)

print("Config saved →", RESULTS_DIR)
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU")


## B. Model Loading


In [ ]:
from transformers import Qwen2_5OmniForConditionalGeneration, Qwen2_5OmniProcessor
from qwen_omni_utils import process_mm_info

print(f"Loading {MODEL_ID} ...")

processor = Qwen2_5OmniProcessor.from_pretrained(MODEL_ID)

model = Qwen2_5OmniForConditionalGeneration.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.bfloat16,
    device_map="auto",
    attn_implementation="sdpa",  # Use SDPA instead of flash_attention_2 for broader GPU support
)
model.disable_talker()  # We only need text output, not audio generation

print("✅ Model loaded successfully.")


## B.1 — Quick Sanity Check


In [ ]:
import librosa, glob

audio_files = sorted(glob.glob(f"{AUDIO_BASE}/clean_nepali_200_flat/*.wav"))
if not audio_files:
    audio_files = sorted(glob.glob(f"{AUDIO_BASE}/clean_nepali_200_flat/*.mp3"))
test_audio_path = audio_files[0]
print(f"Testing: {test_audio_path}")

# Qwen2.5-Omni uses chat messages with audio_url
conversation = [
    {
        "role": "system",
        "content": [
            {"type": "text", "text": "You are a helpful assistant."}
        ]
    },
    {
        "role": "user",
        "content": [
            {"type": "audio", "audio": test_audio_path},
            {"type": "text", "text": "Transcribe the following Nepali speech verbatim in Devanagari script. Output ONLY the transcription, nothing else."},
        ],
    },
]

# Process multimodal info
text = processor.apply_chat_template(conversation, tokenize=False, add_generation_prompt=True)
audios, images, videos = process_mm_info(conversation, use_audio_in_video=True)

inputs = processor(
    text=text,
    audio=audios,
    images=images,
    videos=videos,
    padding=True,
    return_tensors="pt",
)
inputs = inputs.to(model.device).to(model.dtype)

with torch.no_grad():
    output_ids = model.generate(**inputs, max_new_tokens=256, do_sample=False)

new_tokens = output_ids[:, inputs["input_ids"].shape[1]:]
result = processor.batch_decode(new_tokens, skip_special_tokens=True)[0]
print(f"\n📝 Model output:\n{result}")


## C. Prompt Templates


In [ ]:
PROMPTS = {
    "L0_a": "Transcribe the following speech segment in its original language. Only output the transcription.",
    "L1_a": (
        "You are a speech transcription system. "
        "Transcribe the following Nepali audio into Nepali text using Devanagari script. "
        "Produce a verbatim transcription. Do not translate. "
        "Return only the transcription, nothing else."
    ),
    "L1_b": (
        "Task: verbatim Nepali speech transcription.\n"
        "Language: Nepali (Devanagari script).\n"
        "Instructions: transcribe exactly what is spoken. Do not translate. "
        "Output only the transcription."
    ),
    "L2_a": (
        "Transcribe the spoken Nepali audio verbatim in Devanagari script. "
        "Preserve any English words in Latin script. "
        "Maintain the order of Nepali–English code-switching as spoken. "
        "Do not translate between languages. "
        "Keep fillers, repetitions, corrections, and incomplete words. "
        "Do not correct grammar. Do not infer inaudible words. "
        "Do not add timestamps, speaker labels, explanations, or confidence scores. "
        "Return only the transcription."
    ),
    "L2_b": (
        "You are a verbatim transcription system for Nepali speech.\n"
        "Rules:\n"
        "1. Write Nepali in Devanagari.\n"
        "2. Write English words in Latin script.\n"
        "3. Preserve code-switching order.\n"
        "4. Do not translate.\n"
        "5. Keep fillers, repetitions, corrections, incomplete words.\n"
        "6. Do not correct grammar.\n"
        "7. Do not guess inaudible words.\n"
        "8. No timestamps, no speaker labels, no explanations.\n"
        "9. Output only the transcription."
    ),
}
print(f"Defined {len(PROMPTS)} prompt variants.")


## D. Build Manifest


In [ ]:
import pandas as pd

def build_manifest(audio_dir, condition, max_files=None):
    '''Scan audio directory and build a manifest DataFrame, loading references from CSV if available.'''
    import glob, os
    
    # Try to load metadata
    metadata_df = None
    for meta_name in ["metadata.csv", "noisy_metadata.csv"]:
        meta_path = os.path.join(audio_dir, meta_name)
        if os.path.exists(meta_path):
            metadata_df = pd.read_csv(meta_path)
            # Ensure we have a consistent identifier to join on
            if "file" in metadata_df.columns:
                metadata_df["utterance_id"] = metadata_df["file"].apply(lambda x: os.path.splitext(os.path.basename(x))[0])
            break
            
    files = sorted(glob.glob(f"{audio_dir}/**/*.wav", recursive=True)) +             sorted(glob.glob(f"{audio_dir}/**/*.mp3", recursive=True))
            
    if max_files:
        files = files[:max_files]
        
    records = []
    for fp in files:
        uid = os.path.splitext(os.path.basename(fp))[0]
        
        # Look up reference
        ref_text = ""
        if metadata_df is not None and "utterance_id" in metadata_df.columns:
            match = metadata_df[metadata_df["utterance_id"] == uid]
            if not match.empty:
                # Use label_normalized if available, else reference
                if "label_normalized" in match.columns:
                    ref_text = str(match.iloc[0]["label_normalized"])
                elif "reference" in match.columns:
                    ref_text = str(match.iloc[0]["reference"])
                    
        records.append({
            "utterance_id": uid,
            "audio_path": fp,
            "speech_condition": condition,
            "reference_raw": ref_text,
        })
    return pd.DataFrame(records)


manifest_clean = build_manifest(f"{AUDIO_BASE}/clean_nepali_200_flat", "clean", max_files=5)
manifest_noisy = build_manifest(f"{AUDIO_BASE}/noisy_nepali_200", "noisy", max_files=5)
manifest_cs    = build_manifest(f"{AUDIO_BASE}/codeswitched_nepali_200_flat", "codeswitched", max_files=5)
manifest = pd.concat([manifest_clean, manifest_noisy, manifest_cs], ignore_index=True)
print(f"Pilot manifest: {len(manifest)} utterances")


## E. Batch Inference Pipeline


In [ ]:
import time, traceback
import jsonlines
from tqdm.auto import tqdm

def transcribe_one(audio_path, prompt_text):
    conversation = [
        {"role": "system", "content": [{"type": "text", "text": "You are a helpful assistant."}]},
        {"role": "user", "content": [
            {"type": "audio", "audio": audio_path},
            {"type": "text", "text": prompt_text},
        ]},
    ]
    
    text = processor.apply_chat_template(conversation, tokenize=False, add_generation_prompt=True)
    audios, images, videos = process_mm_info(conversation, use_audio_in_video=True)
    
    inputs = processor(text=text, audio=audios, images=images, videos=videos, padding=True, return_tensors="pt")
    inputs = inputs.to(model.device).to(model.dtype)
    
    with torch.no_grad():
        output_ids = model.generate(**inputs, max_new_tokens=experiment_config["max_new_tokens"], do_sample=False)
    
    new_tokens = output_ids[:, inputs["input_ids"].shape[1]:]
    return processor.batch_decode(new_tokens, skip_special_tokens=True)[0]

def clean_output(raw):
    text = raw.strip()
    for prefix in ["Transcription:", "Output:", "```", "**"]:
        if text.startswith(prefix):
            text = text[len(prefix):]
    return text.strip("`*\n ")

def run_pipeline(manifest_df, prompts_dict, output_file):
    output_path = f"{RESULTS_DIR}/{output_file}"
    completed = set()
    if os.path.exists(output_path):
        with jsonlines.open(output_path) as reader:
            for obj in reader:
                completed.add((obj["utterance_id"], obj["prompt_id"]))
        print(f"Resuming: {len(completed)} already completed.")
    
    total = len(manifest_df) * len(prompts_dict)
    pbar = tqdm(total=total, desc="Inference")
    
    for _, row in manifest_df.iterrows():
        for prompt_id, prompt_text in prompts_dict.items():
            key = (row["utterance_id"], prompt_id)
            if key in completed:
                pbar.update(1)
                continue
            record = {
                "model_id": MODEL_ID, "utterance_id": row["utterance_id"],
                "prompt_id": prompt_id, "prompt_level": prompt_id.split("_")[0],
                "audio_path": row["audio_path"], "speech_condition": row["speech_condition"],
                "reference_raw": row.get("reference_raw", ""),
                "status": "success", "raw_output": "", "cleaned_prediction": "",
                "inference_seconds": 0, "timestamp": datetime.now().isoformat(),
            }
            try:
                t0 = time.time()
                raw = transcribe_one(row["audio_path"], prompt_text)
                record["inference_seconds"] = round(time.time() - t0, 2)
                record["raw_output"] = raw
                record["cleaned_prediction"] = clean_output(raw)
                if not record["cleaned_prediction"]:
                    record["status"] = "empty_output"
            except torch.cuda.OutOfMemoryError:
                record["status"] = "out_of_memory"
                gc.collect(); torch.cuda.empty_cache()
            except Exception as e:
                record["status"] = "inference_error"
                record["raw_output"] = str(e)
            
            with jsonlines.open(output_path, mode="a") as writer:
                writer.write(record)
            completed.add(key)
            pbar.update(1)
    pbar.close()
    print(f"\n✅ Done. {len(completed)} results → {output_path}")
    return output_path


## F. Run Inference


In [ ]:
results_file = run_pipeline(manifest, PROMPTS, "raw_predictions.jsonl")


## G. Compute Metrics


In [ ]:
from jiwer import wer, cer
import pandas as pd

def compute_metrics(ref, hyp):
    if not ref or not hyp:
        return {"wer": 1.0, "cer": 1.0}
    try:
        return {"wer": round(wer(ref, hyp), 4), "cer": round(cer(ref, hyp), 4)}
    except:
        return {"wer": 1.0, "cer": 1.0}

results = []
with jsonlines.open(f"{RESULTS_DIR}/raw_predictions.jsonl") as reader:
    for obj in reader:
        if obj["status"] == "success" and obj.get("reference_raw"):
            obj.update(compute_metrics(obj["reference_raw"], obj["cleaned_prediction"]))
        results.append(obj)

df = pd.DataFrame(results)
df.to_csv(f"{RESULTS_DIR}/utterance_metrics.csv", index=False)

if "wer" in df.columns:
    summary = df[df["status"]=="success"].groupby("prompt_id").agg(
        avg_wer=("wer","mean"), avg_cer=("cer","mean"), count=("utterance_id","count")
    ).reset_index().sort_values("avg_wer")
    summary.to_csv(f"{RESULTS_DIR}/prompt_summary.csv", index=False)
    display(summary)
else:
    print("Status distribution:"); print(df["status"].value_counts())
